In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Text Pre-Processing

### 1. Converting to lowercase

In [7]:
df["review"] = df["review"].str.lower()

### 2. Removing the URL's

In [8]:
import re   #regular expression lib

def remove_urls(text):  # created a fnx for removing chars
    text = re.sub( r"https\S+", "", text)  # (pattern, repl, String) is the flow where \S is for non-white space char(no spaces only char) and \s is for white space char(no char only spaces )
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [9]:
def remove_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punc)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Removing HTML tags

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing stopwords

In [11]:
import nltk 

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\SYED
[nltk_data]     MAAZ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\SYED
[nltk_data]     MAAZ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\SYED
[nltk_data]     MAAZ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)  # eg, i/p = "my name is syed maaz", o/p = "my", "name", "is", "syed", "maaz"
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [13]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)  # to convert list to str
    
df["review"] = df["review"].apply(stemming)
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]

y.head()

0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

### 8. Vectorization

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(max_features=5000)  # max_features bcoz our system will crash for big values

X = tf_idf.fit_transform(df["review"])

# Dataset and Dataloader

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248313 stored elements and shape (39665, 5000)>

In [24]:
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 808823 stored elements and shape (9917, 5000)>

In [37]:
# converted it back to arr
X_train = X_train.toarray()
X_test = X_test.toarray()

In [39]:
X_train

array([[0.2133337, 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ]], shape=(39665, 5000))

In [40]:
X_test

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.06314283, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(9917, 5000))

In [42]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [45]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

In [47]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [51]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

# RNN

In [63]:
import torch.nn as nn
import torch.optim as optim

In [68]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_layers=1, neurons=128):
        super().__init__()

        self.hidden_layers = hidden_layers
        self.neurons = neurons

        # RNN layer
        self.rnn = nn.RNN(input_size, neurons, hidden_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(neurons, 1) # i/p features, o/p features (converts 128 neurons to 1)

    def forward(self, x):
        # optional => init hidden_state with 0 check notes (num of hidden layers, batch size, neurons)
        h0 = torch.zeros(self.hidden_layers, x.size(0), self.neurons)
        
        # 1st value = hidden state of all the timesteps => (batch, seq_len, neurons)
        # 2nd value = final hidden state of last timestep (_ we ignore this)
        output, _ = self.rnn(x, h0)

        output = self.fc(output[:, -1, :]) # take all batches, last time step bcoz it contains the info of whole sequence, all neurons 
        return output

    

In [69]:
input_size = X_train.shape[1]  # GPT this line

model = RNN(input_size)

criterion = nn.BCELoss() # because it is a bin classification
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [70]:
# squeeze => remove dimension
# unsquueze => add dimension

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for xb, yb in train_dataloader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) # out model needs 3d but we have 2d so we added 1 dim to make it 3d
        outputs = model(xb) # this is 2d (batch_sise, 1) but for sigmoid we need 1d
        outputs = torch.sigmoid(outputs.squeeze()) # this is now 1d (batch_size, )

        loss = criterion(outputs, yb) # compute loss
        loss.backward()
        optimizer.step() # update weights

        epoch_training_loss += loss.item()
    print(f"Epoch: {epoch} / {epochs} & Loss: {epoch_training_loss / len(train_dataloader)}")

Epoch: 0 / 10 & Loss: 0.37797993380696543
Epoch: 1 / 10 & Loss: 0.2640992017763276
Epoch: 2 / 10 & Loss: 0.248888623894703
Epoch: 3 / 10 & Loss: 0.24171221841487192
Epoch: 4 / 10 & Loss: 0.23737460110456712
Epoch: 5 / 10 & Loss: 0.23406052293796692
Epoch: 6 / 10 & Loss: 0.2319986630591654
Epoch: 7 / 10 & Loss: 0.2302266475894759
Epoch: 8 / 10 & Loss: 0.2285050536235494
Epoch: 9 / 10 & Loss: 0.22731141911879663


In [72]:
# Evaluate

model.eval()

with torch.no_grad():
    total = 0
    correct = 0

    for xb, yb in test_dataloader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float() # if predicted is greater than 0.5 convert it to 1

        correct += (predicted == yb).sum().item()
        total += yb.size(0)
    print(f"Accuracy: {correct / total}")

Accuracy: 0.8567106988000404
